# Full-state reach, expected finals, coverage & pair-difference analysis

Everything below is over the full `GameState` (`filled_mask, upper_total, lower_total, num_yahtzees`) under **optimal play**, composed from the already-computed turn kernels + state values (`math/state_reach_coverage.py`). No new game math.

Run the setup cell first (the forward pass takes a few minutes), then explore. Change the `lvl = ...` in a cell to look at a different level.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')                 # run from math/ so the module's data/ paths resolve
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import state_reach_coverage as src

per_level = src.forward_reach(verbose=True)   # heavy: forward pass over the full-state DAG
total = sum(len(d['reach']) for d in per_level)
print()
print(f'{total:,} states reached by optimal play (reach sums to 1 per level above)')

## Coverage — min #states per level to cover each probability threshold

The `TOTAL` row sums the per-level counts; its `n_states` is every state optimal play ever reaches (≈ 21% of the ~105.5M all-play enumeration).

In [ ]:
tbl = src.coverage_table(per_level)
rows = []
for lvl in range(14):
    r = tbl[lvl]
    rows.append(dict(level=lvl, n_states=r['n_states'], **{f'{t*100:g}%': r[t] for t in src.THRESHOLDS}))
t = tbl['total']
rows.append(dict(level='TOTAL', n_states=t['n_states'], **{f'{x*100:g}%': t[x] for x in src.THRESHOLDS}))
pd.DataFrame(rows).set_index('level')

## Expected final points per state

`expected_final = locked + V(reduced projection)`. Sanity check: level 0 (the empty card) should be `V(empty) ≈ 254.6`.

In [ ]:
for lvl in range(14):
    d = per_level[lvl]
    ef, w = d['expected_final'], d['reach']
    print(f'level {lvl:2d}: min {ef.min():7.1f}   reach-mean {np.average(ef, weights=w):7.1f}   max {ef.max():7.1f}')

In [ ]:
lvl = 8
d = per_level[lvl]
plt.figure(figsize=(9, 3))
plt.hist(d['expected_final'], bins=120, weights=d['reach'])
plt.title(f'level {lvl}: reach-weighted distribution of expected final points')
plt.xlabel('expected final points'); plt.ylabel('probability'); plt.show()

## Pairwise difference distribution

Per level, the number of **unordered** state pairs whose **rounded** expected finals differ by exactly `n` (`n = 0` → distinct states with the same rounded value). Total pairs per level = `C(#states, 2)`.

In [ ]:
dd = src.diff_distribution(per_level)
for lvl in range(14):
    d = dd[lvl]
    nz = np.nonzero(d)[0]
    span = int(nz[-1]) if len(nz) else 0
    print(f'level {lvl:2d}: max diff {span:4d}   pairs(0)={int(d[0]):>16,}   total pairs={int(d.sum()):>18,}')

In [ ]:
lvl = 8                          # change to inspect another level
d = dd[lvl]
plt.figure(figsize=(9, 3))
plt.plot(np.arange(len(d)), d)
plt.yscale('log')
plt.title(f'level {lvl}: # state pairs vs |diff of rounded expected final|')
plt.xlabel('n = |difference|'); plt.ylabel('# pairs (log)'); plt.show()

# the full n -> count array (index n) is dd[lvl]; e.g. pairs with |diff| == 5:
print('pairs with |diff| == 5:', f'{int(dd[lvl][5]):,}')

## Explore individual states

The 12 most-reached states at a chosen level, with their reach probability and expected final. `per_level[lvl]` has aligned arrays `mask, upper, lower, num_yahtzees, reach, expected_final` — slice/sort them however you like.

In [ ]:
lvl = 9
d = per_level[lvl]
order = np.argsort(d['reach'])[::-1][:12]
pd.DataFrame(dict(
    mask=[format(int(m), '013b') for m in d['mask'][order]],
    upper=d['upper'][order], lower=d['lower'][order], num_y=d['num_yahtzees'][order],
    reach=d['reach'][order], exp_final=d['expected_final'][order].round(1),
))